In [63]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
.appName("MyApp")\
.master("local[*]")\
.enableHiveSupport()\
.getOrCreate()
sc = spark.sparkContext 

spark

In [5]:
udfDF = spark.range(5).toDF("num")
def pow3(double_value):
    return double_value**3

pow3(2.0)

8.0

In [6]:
from pyspark.sql.functions import udf, col
power3udf = udf(pow3)

In [7]:
from pyspark.sql.functions import col
udfDF.select(power3udf(col("num"))).show(4)


+---------+
|pow3(num)|
+---------+
|        0|
|        1|
|        8|
|       27|
+---------+
only showing top 4 rows



In [16]:
#JOINS
person = spark.createDataFrame([
    (0,"Bill Chambers", 0,[100]),
    (1,"Matei Zaharia", 1,[500, 250, 100]),
    (2,"Michael Armburst", 1,[250, 100])
])\
.toDF("Person_id", "name", "graduate_program", "spark_status")
graduateProgram = spark.createDataFrame([
    (0, "Masters", "school of information", "UC Berkley"),
    (2, "Masters", "EECS", "UC Berkley"),
    (1, "Ph.D" , "EECS", "UC Berkley")
])\
.toDF("id", "degree", "department", "school")
sparkStatus = spark.createDataFrame([
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributors")
])\
.toDF("id", "status")

person.show()
graduateProgram.show()
sparkStatus.show()

+---------+----------------+----------------+---------------+
|Person_id|            name|graduate_program|   spark_status|
+---------+----------------+----------------+---------------+
|        0|   Bill Chambers|               0|          [100]|
|        1|   Matei Zaharia|               1|[500, 250, 100]|
|        2|Michael Armburst|               1|     [250, 100]|
+---------+----------------+----------------+---------------+

+---+-------+--------------------+----------+
| id| degree|          department|    school|
+---+-------+--------------------+----------+
|  0|Masters|school of informa...|UC Berkley|
|  2|Masters|                EECS|UC Berkley|
|  1|   Ph.D|                EECS|UC Berkley|
+---+-------+--------------------+----------+

+---+--------------+
| id|        status|
+---+--------------+
|500|Vice President|
|250|    PMC Member|
|100|  Contributors|
+---+--------------+



In [17]:
#inner join
joinExpression = person["graduate_program"] == graduateProgram["id"]
person.join(graduateProgram, joinExpression).show()

+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|Person_id|            name|graduate_program|   spark_status| id| degree|          department|    school|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|        0|   Bill Chambers|               0|          [100]|  0|Masters|school of informa...|UC Berkley|
|        1|   Matei Zaharia|               1|[500, 250, 100]|  1|   Ph.D|                EECS|UC Berkley|
|        2|Michael Armburst|               1|     [250, 100]|  1|   Ph.D|                EECS|UC Berkley|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+



In [18]:
#Outer Join
joinType = "outer"
person.join(graduateProgram, joinExpression,joinType).show()

+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|Person_id|            name|graduate_program|   spark_status| id| degree|          department|    school|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|        0|   Bill Chambers|               0|          [100]|  0|Masters|school of informa...|UC Berkley|
|        1|   Matei Zaharia|               1|[500, 250, 100]|  1|   Ph.D|                EECS|UC Berkley|
|        2|Michael Armburst|               1|     [250, 100]|  1|   Ph.D|                EECS|UC Berkley|
|     NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkley|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+



In [19]:
joinType = "right_outer"
person.join(graduateProgram, joinExpression, joinType).show()

+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|Person_id|            name|graduate_program|   spark_status| id| degree|          department|    school|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+
|        0|   Bill Chambers|               0|          [100]|  0|Masters|school of informa...|UC Berkley|
|     NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkley|
|        2|Michael Armburst|               1|     [250, 100]|  1|   Ph.D|                EECS|UC Berkley|
|        1|   Matei Zaharia|               1|[500, 250, 100]|  1|   Ph.D|                EECS|UC Berkley|
+---------+----------------+----------------+---------------+---+-------+--------------------+----------+



In [20]:
joinType = "left_outer"
graduateProgram.join(person, joinExpression, joinType).show()

+---+-------+--------------------+----------+---------+----------------+----------------+---------------+
| id| degree|          department|    school|Person_id|            name|graduate_program|   spark_status|
+---+-------+--------------------+----------+---------+----------------+----------------+---------------+
|  0|Masters|school of informa...|UC Berkley|        0|   Bill Chambers|               0|          [100]|
|  2|Masters|                EECS|UC Berkley|     NULL|            NULL|            NULL|           NULL|
|  1|   Ph.D|                EECS|UC Berkley|        2|Michael Armburst|               1|     [250, 100]|
|  1|   Ph.D|                EECS|UC Berkley|        1|   Matei Zaharia|               1|[500, 250, 100]|
+---+-------+--------------------+----------+---------+----------------+----------------+---------------+



In [21]:
joinType = "left_semi"
graduateProgram.join(person, joinExpression, joinType).show()

+---+-------+--------------------+----------+
| id| degree|          department|    school|
+---+-------+--------------------+----------+
|  0|Masters|school of informa...|UC Berkley|
|  1|   Ph.D|                EECS|UC Berkley|
+---+-------+--------------------+----------+



In [22]:
joinType = "left_anti"
graduateProgram.join(person, joinExpression, joinType).show()

+---+-------+----------+----------+
| id| degree|department|    school|
+---+-------+----------+----------+
|  2|Masters|      EECS|UC Berkley|
+---+-------+----------+----------+



In [23]:
joinType = "cross"
graduateProgram.join(person, joinExpression, joinType).show()

+---+-------+--------------------+----------+---------+----------------+----------------+---------------+
| id| degree|          department|    school|Person_id|            name|graduate_program|   spark_status|
+---+-------+--------------------+----------+---------+----------------+----------------+---------------+
|  0|Masters|school of informa...|UC Berkley|        0|   Bill Chambers|               0|          [100]|
|  1|   Ph.D|                EECS|UC Berkley|        1|   Matei Zaharia|               1|[500, 250, 100]|
|  1|   Ph.D|                EECS|UC Berkley|        2|Michael Armburst|               1|     [250, 100]|
+---+-------+--------------------+----------+---------+----------------+----------------+---------------+



In [25]:
csvFile = spark.read.format("csv")\
.option("header", "true")\
.option("mode", "FAILFAST")\
.option("inferSchema","true")\
.load("data/flight-data/csv/2010-summary.csv")

In [26]:
#coverting csv into tsv
csvFile.write.format("csv").mode("overwrite").option("sep","\t").save("/tmp/my-tsv-file.tsv")

In [27]:
csvFile.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|    1|
|       United States|            Ireland|  264|
|       United States|              India|   69|
|               Egypt|      United States|   24|
|   Equatorial Guinea|      United States|    1|
|       United States|          Singapore|   25|
|       United States|            Grenada|   54|
|          Costa Rica|      United States|  477|
|             Senegal|      United States|   29|
|       United States|   Marshall Islands|   44|
|              Guyana|      United States|   17|
|       United States|       Sint Maarten|   53|
|               Malta|      United States|    1|
|             Bolivia|      United States|   46|
|            Anguilla|      United States|   21|
|Turks and Caicos ...|      United States|  136|
|       United States|        Afghanistan|    2|
|Saint Vincent and..

In [30]:
# reading JSON file
spark.read.format("json").option("mode", "FAILFAST").option("inferSchema", "true").load("data/flight-data/json/2010-summary.json").show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [32]:
csvFile.write.format("json").mode("overwrite").save("/tmp/my-json-file.json")

In [34]:
spark.read.format("parquet").load("data/flight-data/parquet/2010-summary.parquet").show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [36]:
csvFile.write.format("parquet").mode("overwrite").save("/tmp/my-parquet-file.parquet")

In [37]:
#ORC LARGE STREAM DATA
spark.read.format("orc").load("data/flight-data/orc/2010-summary.orc").show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows



In [39]:
csvFile.write.format("orc").mode("overwrite").save("/tmp/my-orc-file.orc")

In [41]:
csvFile.repartition(5).write.format("csv").save("/tmp/multiple.csv")

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/tmp/multiple.csv already exists. Set mode as "overwrite" to overwrite the existing path.

In [45]:
csvFile.limit(10).write.mode("overwrite").partitionBy("DEST_COUNTRY_NAME").save("/tmp/partitioned-files.parquet")

In [48]:
 spark.sql("SELECT 1 + 1").show()

+-------+
|(1 + 1)|
+-------+
|      2|
+-------+



In [52]:
spark.read.json("data/flight-data/json/2015-summary.json")\
    .createOrReplaceTempView("some_sql_view")

spark.sql("""
SELECT DEST_COUNTRY_NAME, sum(count)
FROM some_sql_view GROUP BY DEST_COUNTRY_NAME
""").where("DEST_COUNTRY_NAME like 'S%'").where("`sum(count)` > 10").count()

12

In [54]:
spark.sql("""CREATE TABLE flights (
DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count LONG)
USING JSON OPTIONS (path '/data/flight-data/json/2015-summary.json')""")

DataFrame[]

In [60]:
spark.sql("SELECT * FROM flights").show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
+-----------------+-------------------+-----+



In [66]:
spark.sql(""" CREATE EXTERNAL TABLE hive_flights (
    DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count LONG)

ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LOCATION 'data/flight-data-hive/' """)

AnalysisException: [NOT_SUPPORTED_COMMAND_WITHOUT_HIVE_SUPPORT] CREATE Hive TABLE (AS SELECT) is not supported, if you want to enable it, please set "spark.sql.catalogImplementation" to "hive".;
'CreateTable `spark_catalog`.`default`.`hive_flights`, org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe, ErrorIfExists
